Import the files

In [14]:
import pandas as pd
customers = pd.read_csv("/content/olist_customers_dataset.csv")
geolocation = pd.read_csv("/content/olist_geolocation_dataset.csv")
order_items = pd.read_csv("/content/olist_order_items_dataset.csv")
order_payments = pd.read_csv("/content/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("/content/olist_order_reviews_dataset.csv")
orders = pd.read_csv("/content/olist_orders_dataset.csv")
products = pd.read_csv("/content/olist_products_dataset.csv")
sellers = pd.read_csv("/content/olist_sellers_dataset.csv")
product_category = pd.read_csv("/content/product_category_name_translation.csv")

We are going to check the rows , columns , duplicate_data etc

In [15]:
print("Customers:", customers.shape)
print("Geolocation:", geolocation.shape)
print("Order Items:", order_items.shape)
print("Payments:", order_payments.shape)
print("Reviews:", order_reviews.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Category Translation:", product_category.shape)


Customers: (99441, 5)
Geolocation: (1000163, 5)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Orders: (99441, 8)
Products: (32951, 9)
Sellers: (3095, 4)
Category Translation: (71, 2)


In [16]:
print(customers.isnull().sum())
print(geolocation.isnull().sum())
print(order_items.isnull().sum())
print(order_payments.isnull().sum())
print(order_reviews.isnull().sum())
print(orders.isnull().sum())
print(products.isnull().sum())
print(sellers.isnull().sum())
print(product_category.isnull().sum())

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64
order_id                            0
customer_id                         0
order_st

In [17]:
print( customers.duplicated().sum())
print( geolocation.duplicated().sum())
print( order_items.duplicated().sum())
print(order_payments.duplicated().sum())
print( order_reviews.duplicated().sum())
print( orders.duplicated().sum())
print(products.duplicated().sum())
print( sellers.duplicated().sum())
print( product_category.duplicated().sum())


0
261831
0
0
0
0
0
0
0


In [18]:
print("Total geolocation rows:", len(geolocation))

print( geolocation["geolocation_zip_code_prefix"].nunique())


Total geolocation rows: 1000163
19015


In [19]:

print(orders["order_status"].value_counts())

print(
    "\nMissing delivery dates by order status:"
)

print(
    orders.groupby("order_status")[
        "order_delivered_customer_date"
    ].apply(lambda x: x.isna().sum())
)


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Missing delivery dates by order status:
order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_delivered_customer_date, dtype: int64


In [20]:
order_analysis = orders.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

order_analysis = order_analysis.merge(
    order_items[["order_id", "price", "freight_value"]],
    on="order_id",
    how="left"
)



print("Total rows:", len(order_analysis))
print("Unique orders:", order_analysis["order_id"].nunique())

print("\nSample data:")
print(order_analysis[
    ["order_id", "order_status", "customer_state",
     "order_purchase_timestamp", "price", "freight_value"]
].head())

Total rows: 113425
Unique orders: 99441

Sample data:
                           order_id order_status customer_state  \
0  e481f51cbdc54678b7cc49136f2d6af7    delivered             SP   
1  53cdb2fc8bc7dce0b6741e2150273451    delivered             BA   
2  47770eb9100c2d0c44946d9cf07ec65d    delivered             GO   
3  949d5b44dbf5de918fe9c16f97b45f8a    delivered             RN   
4  ad21c59c0840e6cb83a9ceb5573f8159    delivered             SP   

  order_purchase_timestamp   price  freight_value  
0      2017-10-02 10:56:33   29.99           8.72  
1      2018-07-24 20:41:37  118.70          22.76  
2      2018-08-08 08:38:49  159.90          19.22  
3      2017-11-18 19:28:06   45.00          27.20  
4      2018-02-13 21:18:39   19.90           8.72  


In [24]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

order_sales = order_items.groupby("order_id").agg(
    order_value=("price", "sum")
).reset_index()


order_sales = order_sales.merge(
    orders[["order_id", "order_purchase_timestamp", "order_status"]],
    on="order_id",
    how="left"
)


delivered_orders = order_sales[
    order_sales["order_status"] == "delivered"
].copy()


delivered_orders["month"] = (
    delivered_orders["order_purchase_timestamp"].dt.to_period("M")
)


monthly_sales = delivered_orders.groupby("month").agg(
    total_orders=("order_id", "nunique"),
    total_gmv=("order_value", "sum"),
    average_order_value=("order_value", "mean")
).reset_index()

print(monthly_sales.head(10))

     month  total_orders  total_gmv  average_order_value
0  2016-09             1     134.97           134.970000
1  2016-10           265   40325.11           152.170226
2  2016-12             1      10.90            10.900000
3  2017-01           750  111798.36           149.064480
4  2017-02          1653  234223.40           141.695947
5  2017-03          2546  359198.85           141.083602
6  2017-04          2303  340669.68           147.924307
7  2017-05          3546  489338.25           137.997250
8  2017-06          3135  421923.37           134.584807
9  2017-07          3872  481604.52           124.381333


In [25]:
monthly_sales["month"] = monthly_sales["month"].astype(str)

print(monthly_sales.to_string(index=False))


best_month = monthly_sales.loc[
    monthly_sales["total_gmv"].idxmax()
]

print("\nMonth with highest GMV:", best_month["month"])
print("Highest GMV:", round(best_month["total_gmv"], 2))


best_order_month = monthly_sales.loc[
    monthly_sales["total_orders"].idxmax()
]

print("\nMonth with most orders:", best_order_month["month"])
print("Number of orders:", best_order_month["total_orders"])

  month  total_orders  total_gmv  average_order_value
2016-09             1     134.97           134.970000
2016-10           265   40325.11           152.170226
2016-12             1      10.90            10.900000
2017-01           750  111798.36           149.064480
2017-02          1653  234223.40           141.695947
2017-03          2546  359198.85           141.083602
2017-04          2303  340669.68           147.924307
2017-05          3546  489338.25           137.997250
2017-06          3135  421923.37           134.584807
2017-07          3872  481604.52           124.381333
2017-08          4193  554699.70           132.291844
2017-09          4150  607399.67           146.361366
2017-10          4478  648247.65           144.762762
2017-11          7289  987765.37           135.514525
2017-12          5513  726033.19           131.694756
2018-01          7069  924645.00           130.802801
2018-02          6555  826437.13           126.077365
2018-03          7003  95335

In [27]:
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

In [28]:
delivery_data = orders[
    [
        "order_id",
        "order_status",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].copy()


delivery_data = delivery_data[
    (delivery_data["order_status"] == "delivered")
    & (delivery_data["order_delivered_customer_date"].notna())
    & (delivery_data["order_estimated_delivery_date"].notna())
].copy()

delivery_data["delivery_difference_days"] = (
    delivery_data["order_delivered_customer_date"]
    - delivery_data["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)


delivery_data["delivery_result"] = "On time or early"

delivery_data.loc[
    delivery_data["delivery_difference_days"] > 0,
    "delivery_result"
] = "Late"

print("Orders included:", len(delivery_data))
print("\nDelivery performance:")
print(delivery_data["delivery_result"].value_counts())

print("\nAverage delivery difference in days:")
print(round(delivery_data["delivery_difference_days"].mean(), 2))

Orders included: 96470

Delivery performance:
delivery_result
On time or early    88644
Late                 7826
Name: count, dtype: int64

Average delivery difference in days:
-11.18


In [29]:
# Finding the average review score.

order_review_scores = order_reviews.groupby("order_id").agg(
    review_score=("review_score", "mean")
).reset_index()

delivery_reviews = delivery_data.merge(
    order_review_scores,
    on="order_id",
    how="inner"
)
review_comparison = delivery_reviews.groupby("delivery_result").agg(
    total_orders=("order_id", "nunique"),
    average_review_score=("review_score", "mean")
).reset_index()

print(review_comparison.round(2))

    delivery_result  total_orders  average_review_score
0              Late          7661                  2.57
1  On time or early         88163                  4.29


In [30]:
delivery_by_state = delivery_data.merge(
    orders[["order_id", "customer_id"]],
    on="order_id",
    how="left"
)


delivery_by_state = delivery_by_state.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)



state_summary = delivery_by_state.groupby("customer_state").agg(
    total_orders=("order_id", "nunique"),
    late_orders=("delivery_result", lambda x: (x == "Late").sum())
).reset_index()



state_summary["late_delivery_percent"] = (
    state_summary["late_orders"] / state_summary["total_orders"] * 100
)



state_summary = state_summary[
    state_summary["total_orders"] >= 100
]


state_summary = state_summary.sort_values(
    "late_delivery_percent", ascending=False
)

print(state_summary.head(10).round(2).to_string(index=False))

customer_state  total_orders  late_orders  late_delivery_percent
            AL           397           95                  23.93
            MA           717          141                  19.67
            PI           476           76                  15.97
            CE          1279          196                  15.32
            SE           335           51                  15.22
            BA          3256          457                  14.04
            RJ         12350         1664                  13.47
            TO           274           35                  12.77
            PA           946          117                  12.37
            ES          1995          244                  12.23
